In [0]:
import pandas as pd
from pyspark.sql.functions import * 
from pyspark.sql.window import Window 
from datetime import datetime 
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import timedelta

In [0]:

#spark.sql("CREATE VOLUME workspace.default.incremental_demo");

base_path = "/Volumes/workspace/default/incremental_demo"

source_path = f"{base_path}/source"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"

dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(bronze_path)
dbutils.fs.mkdirs(silver_path)

data_day1 = [
    (1, "A", "2024-01-01"),
    (2, "B", "2024-01-01"),
    (3, "C", "2024-01-01"),
    (4, "D", "2024-01-01"),
    (5, "E", "2024-01-01")
]

columns = ["id", "name", "updated_at"]

df_day1 = spark.createDataFrame(data_day1, columns)
df_day1.show() 
df_day1.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day1")

data_day2 = [
    (6, "A2", "2024-01-02"),
    (7, "B2", "2024-01-02"),
    (8, "C2", "2024-01-02"),
    (9, "D2", "2024-01-02"),
    (10, "E2", "2024-01-02"),
    (6, "B2", "2024-01-02"),
    (7, "C2", "2024-01-02"),
    (8, "D2", "2024-01-02"),
    (9, "E2", "2024-01-02")
]

data_day3 = [
    (15, "A3", "2024-01-03"),
    (16, "B3", "2024-01-03"),
    (17, "C3", "2024-01-03"),
    (18, "D3", "2024-01-03"),
    (19, "E3", "2024-01-03"),
    (20, "F3", "2024-01-03"),
    (21, "G3", "2024-01-03"),
    (22, "H3", "2024-01-03"),
    (23, "Z3", "2024-01-03"),
]

columns = ["id", "name", "updated_at"]
df_day2 = spark.createDataFrame(data_day2, columns)
df_day2.show() 
df_day2.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day2")

df_day3 = spark.createDataFrame(data_day3, columns)
df_day3.show() 
df_day3.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day3")

data_day4 = [
    (1, "A_late", "2024-01-01"),
    (2, "B_late", "2024-01-01"),
    (3, "C_late", "2024-01-02"),
    (4, "D_late", "2024-01-02"),
    (24, "E3", "2024-01-04"),
    (25, "F3", "2024-01-04"),
 
]
df_day4 = spark.createDataFrame(data_day4, columns)
df_day4.show() 
df_day4.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day4_latearriving")

data_day5 = [
    (26, "G", "2024-01-05"),
    (27, "H", "2024-01-05")
 
]
data_day5 = spark.createDataFrame(data_day5, columns)
data_day5.show() 
data_day5.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day5")

data_day6 = [
    (26, "G_updated", "2024-01-05"),
    (27, "H_updated", "2024-01-05"),
    (28, "X", "2024-01-06"),
    (29, "Y", "2024-01-06")
 
]
data_day6 = spark.createDataFrame(data_day6, columns)
data_day6.show() 
data_day6.coalesce(1).write.mode("overwrite").parquet(f"{source_path}/day6")

In [0]:
## create watermark or incrementalmarker table 
spark.sql("""
          USE CATALOG workspace 
            """) ;

spark.sql(""" USE SCHEMA default
           """) ;

spark.sql("""
CREATE TABLE IF NOT EXISTS default.watermark_table (
    table_name STRING PRIMARY KEY,
    updated_at DATE
)
USING DELTA
""")
spark.sql("""
INSERT INTO default.watermark_table
VALUES ('incremental_demo', DATE '1900-01-01')
""")

watermark_tble = spark.sql("select * from default.watermark_table")
watermark_tble.show() 

In [0]:
def update_Watermarktable(latest_updte_dt) : 
    spark.sql(f"""
        UPDATE default.watermark_table
        SET updated_at = '{latest_updte_dt}'
        WHERE table_name = 'incremental_demo'
    """)


dbutils.widgets.text("file_source_path", "")
file_source_path = dbutils.widgets.get("file_source_path")
file_name = file_source_path.split('/')[-1]

## read parquet 

watermark_tble = spark.sql("select * from default.watermark_table")

last_update_dt= watermark_tble.filter(col("table_name") == "incremental_demo").selectExpr("max(updated_at)").collect()[0][0]
print(last_update_dt)

df_bronze = spark.read.parquet(file_source_path)

df_bronze_updated = (df_bronze.withColumn("updated_at", to_date(col("updated_at")))
                      .withColumn("Bromze_Load_Date",date_format(current_timestamp(), 'dd-MM-yyyy')))

# df_bronze_lst_update_dt = datetime.strptime(df_bronze.select(max(col('updated_at'))).collect()[0][0], '%Y-%m-%d').date()
df_bronze_lst_update_dt = df_bronze_updated.select(max(col('updated_at'))).collect()[0][0] 
bronze_df = df_bronze_updated.withColumn('Bronze_batch_rows', lit(df_bronze_updated.count()))
bronze_df = bronze_df.withColumn("source_file_name", lit(file_name))
if bronze_df.count() == 0: 
   print("Emoty Dataframe")                                
bronze_df.coalesce(1).write.mode("append").partitionBy("updated_at").format("delta").saveAsTable("default.delta_bronze_inc")
print("Successfully written to the table")


# if  df_bronze_lst_update_dt > last_update_dt : 
#      incremental = df_bronze_updated.filter(col('updated_at') > last_update_dt)
#      bronze_df = incremental.withColumn('Bronze_batch_rows', lit(df_bronze_updated.count()))
#      bronze_df = bronze_df.withColumn("source_file_name", lit(file_name))
#      if bronze_df.count() == 0: 
#          print("Emoty Dataframe")                                
#      bronze_df.coalesce(1).write.mode("append").partitionBy("updated_at").format("delta").saveAsTable("default.delta_bronze_inc")
#      print("Successfully written to the table")
#      update_Watermarktable(df_bronze_lst_update_dt)
# else:
#      print("Not a incremental part ")
        


In [0]:
rollover_days = 30
bronze_df = spark.read.table('default.delta_bronze_inc')
silver_table_name = 'default.delta_silver_inc'
dedup_window = Window.partitionBy("id").orderBy(desc("updated_at"))
bronze_df_dedup = bronze_df.withColumn('rnk', row_number().over(dedup_window))\
                            .filter(col('rnk') == 1)\
                            .drop('rnk')
latest_update_date = bronze_df_dedup.select(max(col('updated_at'))).collect()[0][0] 
print(latest_update_date)
re_process_date = latest_update_date - timedelta(days=rollover_days)
# silver_df = DeltaTable.forName(spark, silver_table_name)
latest_bronze_df = bronze_df_dedup.filter(col('updated_at') >= re_process_date)
latest_bronze_df = latest_bronze_df.withColumn(
    "silver_load_date",
    current_timestamp()
)

latest_bronze_df = latest_bronze_df.withColumn(
        "silver_rows",
        lit(latest_bronze_df.count())
    )
table_exists = spark.catalog.tableExists(
    silver_table_name
)

if not table_exists:

    print("Silver table does not exist. Creating...")
    
    latest_bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table_name)

    print("Silver Delta table created successfully")
else:
    
    silver_df = DeltaTable.forName(
        spark,
        "default.delta_silver_inc"
    )


    (
        silver_df.alias("tgt")
        .merge(
            latest_bronze_df.alias("src"),
            "tgt.id = src.id"
        )

        .whenMatchedUpdate(
            condition="src.updated_at >= tgt.updated_at",
            set={
                "name": "src.name",
                "updated_at": "src.updated_at",
                 "silver_load_date": "src.silver_load_date"
            }
        )
        .whenNotMatchedInsertAll()
        # .whenNotMatchedInsert(
        #     values={
        #         "id": "src.id",
        #         "name": "src.name",
        #         "updated_at": "src.updated_at",
        #         "Bromze_Load_Date": "src.Bromze_Load_Date",
        #         "Bronze_batch_rows": "src.Bronze_batch_rows",
        #         "source_file_name": "src.source_file_name"
        #     }
        # )

        .execute()
    )
    
update_Watermarktable(latest_update_date)

In [0]:
detail_df = spark.sql("DESCRIBE DETAIL default.delta_bronze_inc")
partition_cols = detail_df.select("partitionColumns").collect()[0][0]
if not partition_cols: 
    print("No Partition columns") 
else:
    partition_sizes_df = (
        spark.read.table("default.delta_bronze_inc")
        .select(*partition_cols, "_metadata.file_size", "_metadata.file_path")
        .groupBy(*partition_cols)
        .agg(
            F.sum("_metadata.file_size").alias("size_bytes"),
            F.countDistinct("_metadata.file_path").alias("file_count")
        )
        .withColumn("size_mb", F.round(F.col("size_bytes") / (1024 * 1024), 2))
        .drop("size_bytes")
    )

partition_sizes_df.orderBy(col('updated_at').desc()).distinct().display() 

In [0]:
active_files_df = spark.read.table("default.delta_bronze_inc").select("_metadata.file_path").distinct()
display(active_files_df)

In [0]:
%sql
create table default.exams_two (student_id int, subject varchar(20), marks int);

insert into default.exams_two values (1,'Chemistry',91),(1,'Physics',91),(1,'Maths',92)
,(2,'Chemistry',80),(2,'Physics',90)
,(3,'Chemistry',80),(3,'Maths',80)
,(4,'Chemistry',71),(4,'Physics',71)
,(5,'Chemistry',79);

select * from default.exams_two;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
df = spark.sql("select * from default.exams_two")
df1 = df.alias('df1')
df_flterd = df.filter(col('subject').isin(['Chemistry','Physics']))
df_flterd = df_flterd.alias('df_flterd')
window_spec = Window.partitionBy('student_id').orderBy(desc('marks'))
# df_flterd.withColumn('previous_marks', lag(col("marks"),1).over(window_spec)).filter(col('previous_marks') == col('marks')).select(col('student_id')).distinct().display() 
df_flterd.display() 
# df.join(df1, df.)